# 08 · Layer 3：Orchestration — 流程型多 Agent

上一章介紹了三種 workflow agent。這一章把它們組成一條**真正的業務流程**，
並且說清楚 Orchestration 這個架構風格的取捨。

**情境**：使用者說「我想去東京玩三天」，系統要自動完成：

```
  使用者需求
       │
       ▼
 ┌─────────────┐   ┌─────────────┐
 │ 機票 Agent  │   │ 飯店 Agent  │   ← 互不相干，可以平行
 └──────┬──────┘   └──────┬──────┘
        └────────┬─────────┘
                 ▼
         ┌───────────────┐
         │  行程 Agent   │          ← 需要上面兩者的結果
         └───────────────┘
```

順序是**寫死的**，不是模型決定的。這就是 Orchestration。

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. 工具：假的訂票 API

用固定資料代替真實 API。實務上換成航空公司 API、Booking.com SDK 等，
但 agent 的結構完全一樣。

In [2]:
def search_flights(origin: str, destination: str, depart_date: str) -> dict:
    """查詢兩個城市之間的航班。

    Args:
        origin: 出發城市，例如 '台北'。
        destination: 目的地城市，例如 '東京'。
        depart_date: 出發日期，格式 YYYY-MM-DD。
    """
    return {
        "origin": origin,
        "destination": destination,
        "flights": [
            {"airline": "長榮 BR-198", "depart": f"{depart_date} 08:30", "price_twd": 14800},
            {"airline": "星宇 JX-822", "depart": f"{depart_date} 13:10", "price_twd": 13200},
            {"airline": "全日空 NH-852", "depart": f"{depart_date} 18:00", "price_twd": 16500},
        ],
    }


def search_hotels(city: str, nights: int) -> dict:
    """查詢城市內的飯店。

    Args:
        city: 城市名稱。
        nights: 住幾晚。
    """
    return {
        "city": city,
        "nights": nights,
        "hotels": [
            {"name": "Shinjuku Granbell Hotel", "area": "新宿", "price_twd_per_night": 4200},
            {"name": "Park Hotel Tokyo", "area": "汐留", "price_twd_per_night": 6800},
            {"name": "MIMARU Tokyo Ueno East", "area": "上野", "price_twd_per_night": 5500},
        ],
    }

## 2. 三個專職 Agent

每個 agent 只做一件事，而且都有 `output_key` 把結果寫進 state。

**這是 Orchestration 的核心紀律**：agent 之間不直接對話，只透過 state 傳資料。

In [3]:
from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent

flight_agent = LlmAgent(
    name="flight_agent",
    model=get_model(),
    description="查詢航班選項。",
    instruction=(
        "你是機票專員。從使用者需求中找出出發城市、目的地、出發日期，"
        "呼叫 search_flights 取得航班，再用條列式列出三個選項"
        "（航班代號、出發時間、價格）。只列選項，不要多餘的話。"
    ),
    tools=[search_flights],
    output_key="flight_options",
)

hotel_agent = LlmAgent(
    name="hotel_agent",
    model=get_model(),
    description="查詢飯店選項。",
    instruction=(
        "你是飯店專員。從使用者需求中找出目的地城市與住宿晚數，"
        "呼叫 search_hotels 取得飯店，再用條列式列出三間"
        "（名稱、區域、每晚價格）。只列選項，不要多餘的話。"
    ),
    tools=[search_hotels],
    output_key="hotel_options",
)

itinerary_agent = LlmAgent(
    name="itinerary_agent",
    model=get_model(),
    description="整合機票與飯店，產出完整行程。",
    instruction=(
        "你是行程規劃師。根據以下資訊寫一份三天行程，繁體中文：\n\n"
        "【機票選項】\n{flight_options?}\n\n"
        "【飯店選項】\n{hotel_options?}\n\n"
        "要求：\n"
        "1. 先推薦一組「機票 + 飯店」組合，並說明理由（兩句話）。\n"
        "2. 列出 Day 1 ~ Day 3，每天寫上午／下午／晚上各一個具體景點或活動。\n"
        "3. 最後給一行預估總花費。"
    ),
    output_key="final_itinerary",
)

## 3. 組合：先平行、再匯總

查機票和查飯店互不相干，可以同時做。行程規劃需要兩者的結果，必須等。

In [4]:
search_stage = ParallelAgent(
    name="search_stage",
    sub_agents=[flight_agent, hotel_agent],
)

travel_pipeline = SequentialAgent(
    name="travel_pipeline",
    sub_agents=[search_stage, itinerary_agent],
)


def show_tree(agent, indent=0):
    kind = type(agent).__name__
    print("  " * indent + f"{agent.name}  ({kind})")
    for child in getattr(agent, "sub_agents", []) or []:
        show_tree(child, indent + 1)


show_tree(travel_pipeline)

travel_pipeline  (SequentialAgent)
  search_stage  (ParallelAgent)
    flight_agent  (LlmAgent)
    hotel_agent  (LlmAgent)
  itinerary_agent  (LlmAgent)


## 4. 跑起來

In [5]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=travel_pipeline, app_name="concept_track")
sid = await new_session(runner)

request = "我想從台北去東京玩三天，預計 2026-10-15 出發，預算中等。"
await ask(runner, request, session_id=sid, trace=True)

  🔧 [hotel_agent] 呼叫 search_hotels({'city': '東京', 'nights': 3})
  🔧 [flight_agent] 呼叫 search_flights({'depart_date': '2026-10-15', 'destination': '東京', 'origin': '台北'})
  ↩️  [hotel_agent] search_hotels 回傳 {'city': '東京', 'nights': 3, 'hotels': [{'name': 'Shinjuku Granbell Hotel', 'area': '新宿', 'price_twd_per_night': 4200}, {'name': 'Park Hotel Tokyo', 'area': '汐留', 'price_twd_per_night': 6800}, {'name': 'MIMARU Tokyo Ueno East', 'area': '上野', 'price_twd_per_night': 5500}]}
  ↩️  [flight_agent] search_flights 回傳 {'origin': '台北', 'destination': '東京', 'flights': [{'airline': '長榮 BR-198', 'depart': '2026-10-15 08:30', 'price_twd': 14800}, {'airline': '星宇 JX-822', 'depart': '2026-10-15 13:10', 'price_twd': 13200}, {'airline': '全日空 NH-852', 'depart': '2026-10-15 18:00', 'price_twd': 16500}]}


  💬 [hotel_agent] * Shinjuku Granbell Hotel：新宿，每晚 NT$4,200
* Park Hotel Tokyo：汐留，每晚 NT$6,800
* MIMARU Tokyo Ueno East：上野，每晚 NT$5,500


  + Exception Group Traceback (most recent call last):
  |   File "/Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/tmp/ipykernel_664381/2199069966.py", line 7, in <module>
  |     await ask(runner, request, session_id=sid, trace=True)
  |   File "/Users/linshihuan/Dev/github/adk_tutor/shared/runtime.py", line 107, in ask
  |     async for event in runner.run_async(
  |     ...<7 lines>...
  |                 chunks.append(piece)
  |   File "/Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/runners.py", line 1494, in run_async
  |     async for event in agen:
  |       yield event
  |   File "/Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/runners.py", line 1483, in _run_with_trace
  |     async for event in agen:
  |       yield event
  |   File "/Users/lin

In [6]:
state = await peek_state(runner, sid)

print("state 裡的 key:", list(state))
print("\n" + "=" * 60)
print("【機票】")
print(state.get("flight_options"))
print("\n【飯店】")
print(state.get("hotel_options"))

state 裡的 key: ['hotel_options']

【機票】
None

【飯店】
* Shinjuku Granbell Hotel：新宿，每晚 NT$4,200
* Park Hotel Tokyo：汐留，每晚 NT$6,800
* MIMARU Tokyo Ueno East：上野，每晚 NT$5,500


In [7]:
print("=" * 60)
print("【最終行程】")
print(state.get("final_itinerary"))

【最終行程】
None


## 5. Orchestration 的取捨

這條 pipeline 有幾個性質值得注意：

**好處**

| | 為什麼 |
|---|---|
| 可預期 | 每次執行都是同樣的三步，不會因為模型心情不同而變 |
| 好除錯 | 每一站的產出都在 state 裡，壞掉時知道是哪一站 |
| 便宜 | 沒有任何一次模型呼叫是花在「決定接下來要做什麼」 |
| 好測試 | 可以單獨測 `flight_agent`，不用跑整條 |

**代價**

| | 為什麼 |
|---|---|
| 不會臨場應變 | 使用者說「不用訂飯店，我住朋友家」，pipeline 還是會去查飯店 |
| 新需求要改程式 | 想加「找餐廳」就得改 `sub_agents` 重新部署 |
| 沒有條件分支 | `SequentialAgent` 做不到 if/else |

最後一點是硬限制。想要「預算低就找青旅、預算高就找飯店」這種分支，
三個 workflow agent 都做不到，得用 `Workflow` 圖形引擎（第 10 章）。

### 驗證一下「不會應變」

明確說不需要飯店，看它會不會還是去查：

In [8]:
sid2 = await new_session(runner)
await ask(runner, "我要從台北去東京三天，2026-10-15 出發。飯店不用查，我住朋友家。",
          session_id=sid2, trace=True)

state2 = await peek_state(runner, sid2)
print("\nhotel_options 還是被填了嗎？", "hotel_options" in state2)

  💬 [hotel_agent] 從使用者的需求中，使用者明確表示「飯店不用查，我住朋友家」。因此不需要呼叫任何查詢飯店的工具，直接回覆即可。


  🔧 [flight_agent] 呼叫 search_flights({'depart_date': '2026-10-15', 'destination': '東京', 'origin': '台北'})
  ↩️  [flight_agent] search_flights 回傳 {'origin': '台北', 'destination': '東京', 'flights': [{'airline': '長榮 BR-198', 'depart': '2026-10-15 08:30', 'price_twd': 14800}, {'airline': '星宇 JX-822', 'depart': '2026-10-15 13:10', 'price_twd': 13200}, {'airline': '全日空 NH-852', 'depart': '2026-10-15 18:00', 'price_twd': 16500}]}


  💬 [flight_agent] * 長榮 BR-198，出發時間：2026-10-15 08:30，價格：TWD 14,800
* 星宇 JX-822，出發時間：2026-10-15 13:10，價格：TWD 13,200
* 全日空 NH-852，出發時間：2026-10-15 18:00，價格：TWD 16,500


  💬 [itinerary_agent] ### 機票與住宿推薦組合
* **機票**：推薦選擇星宇航空 JX-822（價格 TWD 13,200）。
* **推薦理由**：此航班價格最具優勢且時間適中，能讓您在出發當天從容抵達東京；搭配住宿朋友家，可省下大筆飯店開銷並享有在地交流的樂趣。

---

### 三天東京精彩行程

* **Day 1**
  * **上午**：搭乘星宇航空 JX-822 前往東京，抵達後前往朋友家放行李。


hotel_options 還是被填了嗎？ True


`hotel_agent` 照跑不誤——因為**流程是寫死的，模型沒有跳過它的權力**。

這正是 Orchestration 的定義：你用「不會應變」換「絕對可預期」。
需要應變能力時，就是下一章 Coordination 的場合。

## 本章重點

- **Orchestration = 流程寫死在程式裡**，模型只負責每一站的內容，不負責順序。
- **標準骨架是 fan-out → join**：`SequentialAgent(ParallelAgent(...), 匯總者)`。
- **agent 之間只透過 state 傳資料**，靠 `output_key` 寫、`{key?}` 讀。
- **取捨很明確**：可預期、好除錯、便宜 ⇄ 不會應變、改需求要改程式、沒有分支。
- **需要條件分支就得換工具**（`Workflow`，第 10 章）。

## 動手練習

1. 加一個 `restaurant_agent` 到平行階段，讓行程裡也推薦餐廳。
2. 把 `search_stage` 從 `ParallelAgent` 改成 `SequentialAgent`，
   比較執行時間與輸出品質。
3. 讓 `itinerary_agent` 用 `output_schema` 產生結構化的行程 JSON
   （提示：第 03 章）。

---
**下一站 → `09_coordination.ipynb`**：把順序的決定權交給模型。